In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from lic3D import runlic_3d
import nibabel as nib
from pathlib import Path
import time

# LIC

In [ ]:
def generate_lic_from_peaks(peaks_path: Path, output_path: Path, L = 10, texture: np.ndarray = None, magnitude = True):

    # Load peaks and get metadata
    peaks_img = nib.load(str(peaks_path))
    peaks = peaks_img.get_fdata(dtype=np.float32)
    affine = peaks_img.affine

    # Rotate vectors to voxel space
    voxel_to_world = affine[:3, :3]
    world_to_voxel = np.linalg.inv(voxel_to_world)
    rotated = np.einsum('ij,xyzj->xyzi', world_to_voxel, peaks)

    # Split into components
    U, V, W = rotated[..., 0], rotated[..., 1], rotated[..., 2]

    # Replace NaNs
    nan_replacement = 0
    U = np.nan_to_num(U, nan=nan_replacement)
    V = np.nan_to_num(V, nan=nan_replacement)
    W = np.nan_to_num(W, nan=nan_replacement)

    # Apply LIC
    lic_map = runlic_3d(U, V, W, L=L, magnitude=magnitude, texture=texture)

    # Save result
    lic_img = nib.Nifti1Image(lic_map, affine)
    nib.save(lic_img, str(output_path))

In [ ]:
### Generate LIC for main and secondary peaks
### Need to extract peaks from FODs

generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/main_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/LIC_main.nii.gz")
)

generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/secondary_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/LIC_secondary.nii.gz")
)

# RGB LIC

In [ ]:
def generate_rgb_lic_from_peaks(peaks_path: Path,
                                           lic_path: Path,
                                          out_rgb: Path):
    # Load & rotate to voxel space
    img            = nib.load(str(peaks_path))
    peaks          = img.get_fdata(dtype=np.float32)
    affine         = img.affine
    voxel_to_world = affine[:3, :3]
    world_to_voxel = np.linalg.inv(voxel_to_world)
    rotated        = np.einsum('ij,xyzj->xyzi', world_to_voxel, peaks)
    U, V, W        = rotated[...,0], rotated[...,1], rotated[...,2]

    # Clean NaNs (and zero out any truly zero vectors)
    U = np.nan_to_num(U, nan=0.0)
    V = np.nan_to_num(V, nan=0.0)
    W = np.nan_to_num(W, nan=0.0)

    # Import grayscale LIC (white noise texture)
    lic_img = nib.load(str(lic_path))
    lic_gray = lic_img.get_fdata(dtype=np.float32)

    # Build orientation color map
    norm = np.sqrt(U**2 + V**2 + W**2) + 1e-12
    R    = np.abs(U) / norm
    G    = np.abs(V) / norm
    B    = np.abs(W) / norm

    # Modulate the gray LIC by the color map
    lic_rgb = np.stack([R * lic_gray,
                        G * lic_gray,
                        B * lic_gray],
                       axis=-1)

    # Save RGB LIC volume
    rgb_img = nib.Nifti1Image(lic_rgb.astype(np.float32), affine)
    nib.save(rgb_img, str(out_rgb))

In [ ]:
### Generate RGB LIC for main and secondary peaks
### Need grayscale LIC maps

generate_rgb_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/main_peaks.nii.gz"),
    lic_path = Path("sub-CON02/x10/INR/LIC_main.nii.gz"),
    out_rgb=Path("sub-CON02/x10/INR/RGB/LIC_main_rgb.nii.gz"),
)

generate_rgb_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/secondary_peaks.nii.gz"),
    lic_path = Path("sub-CON02/x10/INR/LIC_secondary.nii.gz"),
    out_rgb=Path("sub-CON02/x10/INR/RGB/LIC_secondary_rgb.nii.gz"),
)

# Multi-kernel LIC

In [ ]:
def combine_lic_maps(
    main_lic_path: Path,
    second_lic_path: Path,
    output_avg_path: Path,
    output_max_path: Path
):
    # Load both LIC maps
    img_main = nib.load(str(main_lic_path))
    img_second = nib.load(str(second_lic_path))

    # Get image data
    data_main = img_main.get_fdata()
    data_second = img_second.get_fdata()

    # Average
    avg_data = (data_main + data_second) / 2.0
    nib.save(nib.Nifti1Image(avg_data, img_main.affine), str(output_avg_path))

    # Max
    max_data = np.maximum(data_main, data_second)
    nib.save(nib.Nifti1Image(max_data, img_main.affine), str(output_max_path))

In [ ]:
### Generate multi-kernel LIC
### Need grayscale LIC maps

combine_lic_maps(
    main_lic_path=Path("sub-CON02/x10/INR/LIC_main.nii.gz"),
    second_lic_path=Path("sub-CON02/x10/INR/LIC_secondary.nii.gz"),
    output_avg_path=Path("sub-CON02/x10/INR/LIC_avg.nii.gz"),
    output_max_path=Path("sub-CON02/x10/INR/LIC_max.nii.gz")
)

# Multi-kernel RGB LIC

In [8]:
def combine_lic_maps_rgb(
    main_lic_path: Path,
    second_lic_path: Path,
    output_avg_path: Path,
    output_max_path: Path
):
    # Load both LIC maps
    img_main = nib.load(str(main_lic_path))
    img_second = nib.load(str(second_lic_path))

    # Get image data
    data_main = img_main.get_fdata(dtype=np.float32)
    data_second = img_second.get_fdata(dtype=np.float32)

    # Compute average (per channel)
    avg_data = (data_main + data_second) / 2.0

    # Compute per-voxel vector norms (intensity magnitude)
    norm_main = np.linalg.norm(data_main, axis=-1)
    norm_second = np.linalg.norm(data_second, axis=-1)

    # Build mask where main map dominates
    mask = (norm_main >= norm_second)[..., np.newaxis]

    # Choose entire RGB triplet from either main or secondary
    max_data = np.where(mask, data_main, data_second)

    # Save results
    nib.save(nib.Nifti1Image(avg_data.astype(np.float32), img_main.affine), str(output_avg_path))
    nib.save(nib.Nifti1Image(max_data.astype(np.float32), img_main.affine), str(output_max_path))

In [ ]:
### Generate multi-kernel RGB LIC
### Need RGB LIC maps

combine_lic_maps_rgb(
    main_lic_path=Path("sub-CON02/x10/INR/1RGB/LIC_main_rgb.nii.gz"),
    second_lic_path=Path("sub-CON02/x10/INR/1RGB/LIC_secondary_rgb.nii.gz"),
    output_avg_path=Path("sub-CON02/x10/INR/1RGB/LIC_avg_rgb.nii.gz"),
    output_max_path=Path("sub-CON02/x10/INR/1RGB/LIC_max_rgb.nii.gz")
)

# TDI LIC

In [9]:
def place_random_seeds_in_brain(brain_mask, fill_ratio=0.01, seed=None):
    if seed is not None:
        np.random.seed(seed)

    # Find all “inside-brain”
    flat_mask = brain_mask.ravel()
    brain_indices = np.nonzero(flat_mask)[0]

    # Compute how many seeds we want
    n_brain = brain_indices.size
    n_seeds = int(n_brain * fill_ratio)

    # Randomly choose brain indices
    chosen = np.random.choice(brain_indices, size=n_seeds, replace=False)

    # Build the output mask
    flat_seeds = np.zeros_like(flat_mask, dtype=bool)
    flat_seeds[chosen] = True

    return flat_seeds.reshape(brain_mask.shape)


In [ ]:
### Generate seed mask
### Need brain mask and main and secondary peaks in same file

brain_mask_img = nib.load('sub-CON02/x10/INR/brain_mask_x10.nii.gz')
brain_mask     = brain_mask_img.get_fdata().astype(bool)

seeds_mask = place_random_seeds_in_brain(brain_mask,
                                          fill_ratio=0.01,
                                          seed=13)

peaks_img = nib.load('sub-CON02/x10/INR/peaks.nii.gz')

seed_img = nib.Nifti1Image(seeds_mask.astype(np.uint8),
                           affine=peaks_img.affine,
                           header=peaks_img.header)

nib.save(seed_img, 'sub-CON02/x10/INR/inp_tex_den/seed_mask_up.nii.gz')

n_seeds = int(seeds_mask.sum())
print(f"Number of seeds placed: {n_seeds}")
n_brain = int(brain_mask.sum())
print(f"Number of brain voxels: {n_brain}")

In [ ]:
### Generate tractogram
### Create TDI from tractogram

!tckgen sub-CON02/x10/INR/peaks.nii.gz sub-CON02/x10/INR/inp_tex_den/tracks.tck \
  -algorithm FACT \
  -seed_grid_per_voxel sub-CON02/x10/INR/inp_tex_den/seed_mask_up.nii.gz 1 \
  -step  0.25 \
  -maxlength 25 \
  -select 0 \
  -seeds 2837875 \
  -force

!tckmap sub-CON02/x10/INR/inp_tex_den/tracks.tck sub-CON02/x10/INR/inp_tex_den/presence_count.nii.gz \
  -template sub-CON02/x10/INR/peaks.nii.gz \
  -precise                  \
  -contrast tdi             \
  -stat_vox sum \
  -force 

In [ ]:
### Binarize TDI map

img = nib.load('/Users/horiapivniceru/thesis/sub-CON02/x10/INR/inp_tex_den/presence_mask.nii.gz')

data = img.get_fdata(dtype=np.float32)
tex = (data > 0.5).astype(np.uint8)

In [ ]:
### Generate TDI LIC maps and employ multi-kernel approach

generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/main_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_main_inp_tex.nii.gz"),
    texture=tex
)

generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/secondary_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_secondary_inp_tex.nii.gz"),
    texture=tex
)

combine_lic_maps(
    main_lic_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_main_inp_tex.nii.gz"),
    second_lic_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_secondary_inp_tex.nii.gz"),
    output_avg_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_avg_inp_tex.nii.gz"),
    output_max_path=Path("sub-CON02/x10/INR/inp_tex_den/LIC_max_inp_tex.nii.gz")
)

# RGB TDI LIC

In [ ]:
### Generate RGB TDI LIC maps

generate_rgb_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/main_peaks.nii.gz"),
    lic_path = Path("sub-CON02/x10/INR/inp_tex_den/LIC_main_inp_tex.nii.gz"),
    out_rgb=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_main_inp_tex_rgb.nii.gz"),
)

generate_rgb_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/secondary_peaks.nii.gz"),
    lic_path = Path("sub-CON02/x10/INR/inp_tex_den/LIC_secondary_inp_tex.nii.gz"),
    out_rgb=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_secondary_inp_tex_rgb.nii.gz"),
)

combine_lic_maps_rgb(
    main_lic_path=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_main_inp_tex_rgb.nii.gz"),
    second_lic_path=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_secondary_inp_tex_rgb.nii.gz"),
    output_avg_path=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_avg_inp_tex_rgb.nii.gz"),
    output_max_path=Path("sub-CON02/x10/INR/inp_tex_den/RGB/LIC_max_inp_tex_rgb.nii.gz")
)

# dMRI DERIVED INPUT TEXTURES

In [ ]:
### Generate LIC maps using FA map as input texture

img = nib.load("sub-CON02/x10/INR/input_textures/fa/fa.nii.gz")
volume = img.get_fdata()
minv, maxv = np.nanmin(volume), np.nanmax(volume)
normed = (volume - minv) / (maxv - minv)
np.random.seed(13)
noise = np.random.rand(*normed.shape)
alpha = 0.75
input_texture = alpha * normed + (1 - alpha) * noise

generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/main_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_main.nii.gz"),
    texture=input_texture
)
generate_lic_from_peaks(
    peaks_path=Path("sub-CON02/x10/INR/secondary_peaks.nii.gz"),
    output_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_secondary.nii.gz"),
    texture=input_texture
)
combine_lic_maps(
    main_lic_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_main.nii.gz"),
    second_lic_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_secondary.nii.gz"),
    output_avg_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_avg.nii.gz"),
    output_max_path=Path("sub-CON02/x10/INR/input_textures/fa/LIC_fa_max.nii.gz")
)